# Customer360 Retail Analytics
## Bronze Automated Ingestion — Order Items

**Notebook:** `AUTO_08_Order_Items_Ingestion`

**Source:** `s3://olist-retail-project/raw/order_items/`

**Checkpoint:** `s3://olist-retail-project/_checkpoints/order_items_ingestion/`

**Target:** `workspace.bronze.order_items`

**Natural grain:** `(order_id, order_item_id)`

Bronze preserves the raw source representation. Silver performs datatype
conversion and referential-integrity validation.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_PATH = "s3://olist-retail-project/raw/order_items/"
CHECKPOINT_PATH = "s3://olist-retail-project/_checkpoints/order_items_ingestion/"
SCHEMA_LOCATION = "s3://olist-retail-project/_schemas/order_items_ingestion/"
BRONZE_TABLE = "workspace.bronze.order_items"

EXPECTED_COLUMNS = [
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
]

print("Source     :", SOURCE_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Target     :", BRONZE_TABLE)

Source     : s3://olist-retail-project/raw/order_items/
Checkpoint : s3://olist-retail-project/_checkpoints/order_items_ingestion/
Target     : workspace.bronze.order_items


In [0]:
# STEP 1 — Verify existing Bronze target

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"{BRONZE_TABLE} does not exist. "
        "Create/load the existing Bronze baseline before enabling incremental ingestion."
    )

bronze_before_df = spark.table(BRONZE_TABLE)
before_count = bronze_before_df.count()

print(f"Current Bronze Order Items rows : {before_count:,}")

Current Bronze Order Items rows : 112,650


In [0]:
# STEP 2 — Validate existing Bronze contract

if bronze_before_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Existing Bronze Order Items schema does not match the contract.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual  : {bronze_before_df.columns}"
    )

expected_string_types = {
    column: "string"
    for column in EXPECTED_COLUMNS
}

actual_types = dict(bronze_before_df.dtypes)

type_errors = {
    column: {
        "expected": expected_type,
        "actual": actual_types.get(column),
    }
    for column, expected_type in expected_string_types.items()
    if actual_types.get(column) != expected_type
}

if type_errors:
    raise ValueError(
        f"Bronze datatype contract failed: {type_errors}"
    )

print("PASS — Existing Bronze Order Items schema matches the contract.")
bronze_before_df.printSchema()

PASS — Existing Bronze Order Items schema matches the contract.
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [0]:
# STEP 3 — Auto Loader stream

order_items_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.includeExistingFiles", "false")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .load(SOURCE_PATH)
        .select(
            F.col("order_id").cast("string"),
            F.col("order_item_id").cast("string"),
            F.col("product_id").cast("string"),
            F.col("seller_id").cast("string"),
            F.col("shipping_limit_date").cast("string"),
            F.col("price").cast("string"),
            F.col("freight_value").cast("string"),
        )
)

if order_items_stream_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Incoming Order Items schema does not match the Bronze contract.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual  : {order_items_stream_df.columns}"
    )

print("PASS — Auto Loader stream configured for new Order Items files.")
print("PASS — Incoming stream schema matches the Bronze contract.")

PASS — Auto Loader stream configured for new Order Items files.
PASS — Incoming stream schema matches the Bronze contract.


In [0]:
# STEP 4 — Incremental append

query = (
    order_items_stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("PASS — Order Items incremental ingestion completed successfully.")

PASS — Order Items incremental ingestion completed successfully.


In [0]:
# STEP 5 — Validate population

bronze_after_df = spark.table(BRONZE_TABLE)
after_count = bronze_after_df.count()

print(f"Current Bronze Order Items rows : {after_count:,}")

if after_count < before_count:
    raise ValueError(
        "Bronze Order Items population decreased after ingestion."
    )

print("PASS — Bronze Order Items population is non-decreasing.")

Current Bronze Order Items rows : 112,650
PASS — Bronze Order Items population is non-decreasing.


In [0]:
# STEP 6 — Validate schema remains unchanged

if bronze_after_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze Order Items schema changed after ingestion.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual  : {bronze_after_df.columns}"
    )

after_types = dict(bronze_after_df.dtypes)

after_type_errors = {
    column: {
        "expected": "string",
        "actual": after_types.get(column),
    }
    for column in EXPECTED_COLUMNS
    if after_types.get(column) != "string"
}

if after_type_errors:
    raise ValueError(
        f"Bronze Order Items datatypes changed: {after_type_errors}"
    )

print("PASS — Bronze Order Items schema remains unchanged.")

PASS — Bronze Order Items schema remains unchanged.


In [0]:
# STEP 7 — Profile business-key fields

key_profile = bronze_after_df.select(
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0)).alias("null_order_id"),
    F.sum(F.when(F.col("order_item_id").isNull(), 1).otherwise(0)).alias("null_order_item_id"),
    F.sum(F.when(F.trim(F.col("order_id")) == "", 1).otherwise(0)).alias("blank_order_id"),
    F.sum(F.when(F.trim(F.col("order_item_id")) == "", 1).otherwise(0)).alias("blank_order_item_id"),
)

display(key_profile)

null_order_id,null_order_item_id,blank_order_id,blank_order_item_id
0,0,0,0


In [0]:
# STEP 8 — Validate composite business key
#
# Important: the natural grain is (order_id, order_item_id).
# Do NOT test order_id alone for uniqueness.

duplicate_key_groups = (
    bronze_after_df
        .groupBy("order_id", "order_item_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
)

print(
    f"Duplicate (order_id, order_item_id) groups : "
    f"{duplicate_key_groups}"
)

# We fail on duplicate business keys because Silver preserves this grain.
if duplicate_key_groups != 0:
    raise ValueError(
        "Bronze Order Items quality gate failed: "
        "duplicate (order_id, order_item_id) keys found."
    )

print("PASS — Order Items composite business key is unique.")

Duplicate (order_id, order_item_id) groups : 0
PASS — Order Items composite business key is unique.


In [0]:
# STEP 9 — Raw preservation profile

profile_df = bronze_after_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("product_id").isNull(), 1).otherwise(0)).alias("null_product_id"),
    F.sum(F.when(F.col("seller_id").isNull(), 1).otherwise(0)).alias("null_seller_id"),
    F.sum(F.when(F.col("shipping_limit_date").isNull(), 1).otherwise(0)).alias("null_shipping_limit_date"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("freight_value").isNull(), 1).otherwise(0)).alias("null_freight_value"),
)

display(profile_df)

print("PASS — Bronze raw-preservation profiling completed.")

total_rows,null_product_id,null_seller_id,null_shipping_limit_date,null_price,null_freight_value
112650,0,0,0,0,0


PASS — Bronze raw-preservation profiling completed.


In [0]:
# STEP 10 — Confirm target availability

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Bronze target unavailable after ingestion: {BRONZE_TABLE}"
    )

print("PASS — Bronze Order Items table is available after ingestion.")

PASS — Bronze Order Items table is available after ingestion.


In [0]:
print("=" * 72)
print("ORDER ITEMS AUTOMATED INGESTION — SUCCESS")
print("=" * 72)
print(f"Source       : {SOURCE_PATH}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Target       : {BRONZE_TABLE}")
print(f"Rows before  : {before_count:,}")
print(f"Rows after   : {after_count:,}")
print("Mode         : Incremental append")
print("File handling: Auto Loader")
print("Schema mode  : Strict contract validation")
print("Bronze role  : Raw source preservation")
print("=" * 72)

ORDER ITEMS AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/order_items/
Checkpoint   : s3://olist-retail-project/_checkpoints/order_items_ingestion/
Target       : workspace.bronze.order_items
Rows before  : 112,650
Rows after   : 112,650
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict contract validation
Bronze role  : Raw source preservation
